# **09. File Exception Handling**

### **9.1 Introduction to File Exceptions**

**1. Why Do File Exceptions Occur?**

- ***File operations can fail: the file may be missing, locked, or you may not have permission.***

- ***Catch these errors so the program fails gracefully.***

**2. What is an Exception?**

- ***Exceptions (run-time errors) occur during the execution of your program, even if the syntax is perfectly valid.***

- ***If not handled, an exception will cause your program to crash and terminate abruptly.***

**3. Common File Exceptions**

| Exception | When it happens |
|-----------|-----------------|
| `FileNotFoundError` | The file or folder does not exist |
| `PermissionError` | No rights to read or write the file |
| `IsADirectoryError` | A folder was given instead of a file |
| `UnicodeDecodeError` | The file's encoding does not match |

### **9.2 `FileNotFoundError`**

**1. What is `FileNotFoundError`?**

- ***It occurs when you try to open a file that does not exist.***

- ***It is raised in `"r"` and `"r+"` modes if the file is missing.***

```
>>> open("non_existent_file.txt", "r")
FileNotFoundError: [Errno 2] No such file or directory: 'non_existent_file.txt'
```

**2. Handling `FileNotFoundError`**

In [3]:
try:
    file_object_read = open("File/non_existent.txt", "r")
    print("File 'non_existent.txt' opened in read mode.")
    file_object_read.close()
except FileNotFoundError:
    print("Error: The file 'non_existent.txt' was not found. Please check the name and try again.")

Error: The file 'non_existent.txt' was not found. Please check the name and try again.


### **9.3 `FileExistsError`**

**1. What is `FileExistsError`?**

- ***It occurs in exclusive creation mode (`"x"`) when the file already exists.***

- ***This is useful to ensure you are creating a new file.***

In [4]:
for attempt in (1, 2):
    try:
        with open("File/unique_file.txt", "x") as f:
            f.write("Created once.")
        print("Attempt", attempt, ": file created.")
    except FileExistsError:
        print("Attempt", attempt, ": Error - file already exists.")

Attempt 1 : file created.
Attempt 2 : Error - file already exists.


### **9.4 `PermissionError`**

**1. What is `PermissionError`?**

- ***It occurs when you do not have the rights to read or write a file.***

**2. Handling `PermissionError`**

```python
try:
    with open(path) as f:
        return f.read()
except PermissionError:
    print(f"No permission to read '{path}'.")
```

### **9.5 `IsADirectoryError`**

**1. What is `IsADirectoryError`?**

- ***It occurs when a folder is given instead of a file.***

In [ ]:
try:
    with open(".") as f:       # "." is a folder, not a file
        f.read()
except IsADirectoryError:
    print("A folder was given instead of a file.")

**Note:** This example works on Linux and macOS. On Windows, opening a folder raises `PermissionError` instead.

### **9.6 `OSError`**

**1. What is `OSError`?**

- ***`OSError` is the base class for errors related to the operating system and file operations.***

- ***`FileNotFoundError`, `FileExistsError`, `PermissionError` and `IsADirectoryError` are all special kinds of `OSError`.***

- ***Because of this, `except OSError` can catch all of them.***

- ***In Python 3, `IOError` is an alias of `OSError`, so both names mean the same thing.***

In [ ]:
print(issubclass(FileNotFoundError, OSError))
print(issubclass(FileExistsError, OSError))
print(issubclass(PermissionError, OSError))
print(issubclass(IsADirectoryError, OSError))
print(IOError is OSError)

**2. Handling General File Errors**

- ***Use a general `OSError` (or `IOError`) to handle any other possible file-related issues (for example, permission errors).***

- ***Place specific handlers such as `FileNotFoundError` first, and `OSError` after them.***

In [7]:
try:
    with open("File/my_notes.txt", "w") as f:
        f.write("Meeting notes for today.\n")
    print("Data written to my_notes.txt successfully.")
except IOError as e:          # Catch potential errors during file operation
    print(f"Error writing to file: {e}")

Data written to my_notes.txt successfully.


**3. Getting Details from the Error Object**

- ***The error object `e` has useful details: `e.errno` (error number), `e.strerror` (error message) and `e.filename` (file name).***

In [8]:
try:
    open("missing_folder/data.txt", "r")
except FileNotFoundError as e:
    print("Specific handler :", e)
except OSError as e:
    print("General handler  :", e)

try:
    open("missing_folder/data.txt", "r")
except OSError as e:
    print(type(e).__name__)
    print(e.errno)
    print(e.strerror)
    print(e.filename)

Specific handler : [Errno 2] No such file or directory: 'missing_folder/data.txt'
FileNotFoundError
2
No such file or directory
missing_folder/data.txt


### **9.7 `try`, `except`, `else`, `finally`**

**1. Syntax:**

```python
try:
    # Code that might raise an exception
except SpecificException1:
    # Handler for SpecificException1
except SpecificException2:
    # Handler for SpecificException2
else:
    # Code to run if NO exception occurs in the 'try' block
finally:
    # Code that ALWAYS runs, regardless of exceptions
```

**2. Meaning of Each Block**

- ***`try`: This block contains the code that you want to monitor for exceptions.***

- ***`except`: If an exception of the given type occurs within the `try` block, the execution jumps immediately to this block.***

- ***`else`: The code in the `else` block is executed only if the `try` block completes successfully, without any exceptions being raised.***

- ***`finally`: The code in the `finally` block is always executed, regardless of whether an exception occurred or not. It is typically used for cleanup operations, such as closing files.***

**3. Important Notes**

- ***Handle specific exceptions rather than catching all possible errors.***

- ***Place the most specific exception handlers first, followed by more general ones.***

- ***Use `as e` to capture the exception object, which contains useful information about the error.***

**4. Example: Handling File Errors with `finally`**

In [ ]:
def read_file(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        print(f"'{path}' was not found.")
    except PermissionError:
        print(f"No permission to read '{path}'.")
    finally:
        print("Read attempt finished.")

read_file("missing.txt")

**5. Example: `try`-`except`-`else`-`finally`**

In [ ]:
def divide_numbers(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Caught an error: Cannot divide by zero!")
        return None  # Return None to indicate failure
    except TypeError:
        print("Caught an error: Invalid types for division (must be numbers)!")
        return None
    else:  # This block runs ONLY if no exception occurred in the 'try' block
        print("Division successful!")
        return result
    finally:  # This block ALWAYS runs
        print("--- Division attempt finished. ---")

# Test cases
print("Scenario 1: Successful division")
res1 = divide_numbers(10, 2)
if res1 is not None:
    print(f"Result: {res1}\n")

print("Scenario 2: Division by zero")
res2 = divide_numbers(10, 0)
if res2 is not None:
    print(f"Result: {res2}\n")

print("Scenario 3: Type error")
res3 = divide_numbers(10, "two")
if res3 is not None:
    print(f"Result: {res3}\n")